12,  I avsnitt 2.2 “Ett kodexempel från början till slut - Huspriser i Kalifornien” så gås ett komplett kodexempel igenom. <br>
På valideringsdatan fick vi RMSE Random Forest Regression: 52277.96578719621. <br>
Försök få ett bättre resultat genom att exempelvis justera hyperparametrar eller genomföra variabelselektion.

In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

In [14]:
# Här ser vi datatyper och antal icke NULL värden i varje kolumn.
# total_bedrooms saknar 167 värde,och ocean_proximity har datatyp objekt, vilket betyder att det är en kategorisk variabel.

housing_original = pd.read_csv("housing.csv")
housing_original.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  object 
dtypes: float64(9), object(1)
memory usage: 1.6+ MB


In [16]:
housing_original.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [17]:
# Vi får se om det finns fler kategorier i ocean_proximity-kolumnen. 
# Och vi ser att det finns bara fem stycken observationer i ISLAND kategorin.

housing_original['ocean_proximity'].value_counts()

ocean_proximity
<1H OCEAN     9136
INLAND        6551
NEAR OCEAN    2658
NEAR BAY      2290
ISLAND           5
Name: count, dtype: int64

In [18]:
# Rader som har värdet ISLAND tas bort och variabeln är nu uppdaterad för att göra våra analyser mer koncisa.

housing = housing_original[housing_original['ocean_proximity'] != 'ISLAND']
housing['ocean_proximity'].value_counts()

ocean_proximity
<1H OCEAN     9136
INLAND        6551
NEAR OCEAN    2658
NEAR BAY      2290
Name: count, dtype: int64

In [20]:
# Jag tar bort rader med saknade värden i total_bedrooms-kolumnen
housing = housing_original.dropna()
housing.info()

<class 'pandas.core.frame.DataFrame'>
Index: 20433 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20433 non-null  float64
 1   latitude            20433 non-null  float64
 2   housing_median_age  20433 non-null  float64
 3   total_rooms         20433 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20433 non-null  float64
 6   households          20433 non-null  float64
 7   median_income       20433 non-null  float64
 8   median_house_value  20433 non-null  float64
 9   ocean_proximity     20433 non-null  object 
dtypes: float64(9), object(1)
memory usage: 1.7+ MB


In [21]:
# Ocean_proximity är en kategorisk variabel, så vi måste göra en dummy-variable-encoding på den innan det används i modellerna.
# iloc är “select by position (index)” och 9: valdes för att visa bara dummy variablerna.

housing = pd.get_dummies(housing, columns = ['ocean_proximity'], dtype = int, prefix = 'dmy')
housing.iloc[[1, 200, 1000, 1850, 5000], 9:]

,dmy_<1H OCEAN,dmy_INLAND,dmy_ISLAND,dmy_NEAR BAY,dmy_NEAR OCEAN
1,0,0,0,1,0
200,0,0,0,1,0
1006,0,1,0,0,0
1861,0,0,0,0,1
5053,1,0,0,0,0


In [23]:
# Dela upp data i tränings-, validerings- och testdata.

train_full, test = train_test_split(housing, test_size=0.2, random_state=40)
train, val = train_test_split(train_full, test_size=0.25, random_state=36)

train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12259 entries, 16259 to 3236
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           12259 non-null  float64
 1   latitude            12259 non-null  float64
 2   housing_median_age  12259 non-null  float64
 3   total_rooms         12259 non-null  float64
 4   total_bedrooms      12259 non-null  float64
 5   population          12259 non-null  float64
 6   households          12259 non-null  float64
 7   median_income       12259 non-null  float64
 8   median_house_value  12259 non-null  float64
 9   dmy_<1H OCEAN       12259 non-null  int64  
 10  dmy_INLAND          12259 non-null  int64  
 11  dmy_ISLAND          12259 non-null  int64  
 12  dmy_NEAR BAY        12259 non-null  int64  
 13  dmy_NEAR OCEAN      12259 non-null  int64  
dtypes: float64(9), int64(5)
memory usage: 1.4 MB


In [ ]:
# I denna uppgift är 'median_house_value' vår target-variabel, så vi delar upp våra data i X (features) och y (target) för varje dataset.
# Koden blir enklare om jag gör det innan datan delas upp i tränings-, validerings- och testdata, men jag ville testa en annan metod.

X_train, y_train = train.drop(columns=['median_house_value'], axis=1), train['median_house_value']
X_val, y_val = val.drop(columns=['median_house_value'], axis=1), val['median_house_value']
X_test, y_test = test.drop(columns=['median_house_value'], axis=1), test['median_house_value']

print(X_train.shape)
print(y_train.shape)

(12259, 13)
(12259,)


In [25]:
# Vi testar några olika inställningar (hyperparametrar) för modellen för att se vilken som ger bäst resultat.

h_params_list = [
    {"n_estimators": 120, "max_depth": 40},  # 120 träd och max_depth 40
    {"n_estimators": 200, "max_depth": 30}, 
    {"n_estimators": 300, "max_depth": None}, 
    {"n_estimators": 300, "max_depth": 20, "min_samples_split": 5}, 
    {"n_estimators": 400, "max_depth": None, "min_samples_leaf": 3}
]
 
best_rmse = float("inf")  # Detta sätter starting value för bästa RMSE till oändligheten så att alla faktiska RMSE-värden kommer att vara mindre än detta.
best_params = None
best_model = None

In [ ]:
# Här loopar vi igenom varje uppsättning hyperparametrar, tränar en Random Forest-modell, gör prediktioner på valideringsdata och beräknar RMSE.
# RandomForestRegressor används för att skapa en regressionsmodell.
# Här ser vi bättre resultat än den i boken.

for h_params in h_params_list:
    rf = RandomForestRegressor(random_state=42, n_jobs=-1, **h_params)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_val)
    rmse = root_mean_squared_error(y_val, y_pred)
    print(f"Hyperparameters: {h_params}, RMSE: {rmse:.2f}")  # Skriver ut vilka hyperparametrar som användes och tillhörande RMSE

Hyperparameters: {'n_estimators': 120, 'max_depth': 40}, RMSE: 49653.20
Hyperparameters: {'n_estimators': 200, 'max_depth': 30}, RMSE: 49554.89
Hyperparameters: {'n_estimators': 300, 'max_depth': None}, RMSE: 49517.71
Hyperparameters: {'n_estimators': 300, 'max_depth': 20, 'min_samples_split': 5}, RMSE: 49604.16
Hyperparameters: {'n_estimators': 400, 'max_depth': None, 'min_samples_leaf': 3}, RMSE: 49415.00
